## 4. División del conjunto de datos y transformaciones

In [13]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Paths
refined_input = Path("../data/processed/refined_data.parquet")
refined_input_N = Path("../data/processed/refined_data_N.parquet")

# Add src/ to Python path
sys.path.append(str(Path("../src").resolve()))
from data.var_type import split_symbolic_continuous

#### 1) Revisión general

In [2]:
df = pd.read_parquet(refined_input)
dn = pd.read_parquet(refined_input_N)

Primero de todo encuentro las variables numéricas y las categóricas/simbólicas para ambos dataset

In [5]:
symbolic, continuous = split_symbolic_continuous(df)
symbolicN, continuousN = split_symbolic_continuous(dn)

# For dn are the same because the only dif is we add another numeric variables so, symbolic = symbolicN
for col in symbolic:
    print(col, sorted(df[col].unique())[:10])

Protocol [np.int64(0), np.int64(6), np.int64(17)]
Fwd PSH Flags [np.uint32(0), np.uint32(1)]
Fwd URG Flags [np.uint32(0), np.uint32(1)]
FIN Flag Cnt [np.uint32(0), np.uint32(1)]
SYN Flag Cnt [np.uint32(0), np.uint32(1)]
RST Flag Cnt [np.uint32(0), np.uint32(1)]
PSH Flag Cnt [np.uint32(0), np.uint32(1)]
ACK Flag Cnt [np.uint32(0), np.uint32(1)]
URG Flag Cnt [np.uint32(0), np.uint32(1)]
CWE Flag Count [np.uint32(0), np.uint32(1)]
ECE Flag Cnt [np.uint32(0), np.uint32(1)]
Label ['Benign', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DoS attacks-GoldenEye', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce']
Dst Port Cat ['dns', 'ephemeral', 'ftp', 'registered', 'ssh', 'web', 'well_known_other']
attack_group ['Benign', 'Bruteforce', 'DDOS', 'DoS', 'Infiltration']
attack_or_benign ['Attack', 'Benign']


#### 2) División del conjunto de datos

Ahora procedo a dividir el conjunto en un conjunto de train (80%) y uno de test (20%). Luego validaremos los modelos por validación cruzada por lo que no es necesario hacer un conjunto aparte de validación.

In [14]:
# Features and labels
X = df.drop(columns=["Label", "attack_group", "attack_or_benign"])
y = df["attack_or_benign"] #I'll start with the binary one which is the easiest
Xn = dn.drop(columns=["Label", "attack_group", "attack_or_benign"])
yn = dn["attack_or_benign"] #I'll start with the binary one which is the easiest

# df Train (80%) / Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# dn Train (80%) / Test (20%)
X_trainN, X_testN, y_trainN, y_testN = train_test_split(
    Xn,
    yn,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# Shapes
print("Dataset df:")
print("  Train shape:", X_train.shape, y_train.shape)
print("  Test shape:", X_test.shape, y_test.shape)

print("Dataset dn:")
print("  Train shape:", X_trainN.shape, y_trainN.shape)
print("  Test shape:", X_testN.shape, y_testN.shape)

Dataset df:
  Train shape: (2573108, 71) (2573108,)
  Test shape: (643278, 71) (643278,)
Dataset dn:
  Train shape: (2573108, 71) (2573108,)
  Test shape: (643278, 71) (643278,)


#### 3) Transformaciones a los datos en el conjunto de entrenamiento